In [1]:
# Cell 1: Environment Setup (Enhanced for Production Pipeline)

import pandas as pd
import numpy as np
import xgboost as xgb
import logging
import warnings
import ast
import joblib  # CRITICAL: Added for saving model artifacts
import os      # CRITICAL: Added for directory management
import re      # CRITICAL: Added for advanced text cleaning
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer # Added for NLP
from sklearn.metrics.pairwise import cosine_similarity
from geopy.distance import geodesic
from datetime import datetime

# 1. Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

# 2. Suppress Warnings
warnings.filterwarnings('ignore')

# 3. Pandas Display Options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# 4. Define Constants
ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

logger.info(f"Environment initialized.")
logger.info(f"XGBoost Version: {xgb.__version__}")
logger.info(f"Pandas Version: {pd.__version__}")
logger.info(f"Artifacts Directory: {os.path.abspath(ARTIFACTS_DIR)}")

13:40:33 - INFO - Environment initialized.
13:40:33 - INFO - XGBoost Version: 3.1.3
13:40:33 - INFO - Pandas Version: 2.3.3
13:40:33 - INFO - Artifacts Directory: C:\Users\Dell\PF Propertyfinder\Phase 2\New_Files\artifacts


In [2]:
pip install geopy

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Cell 2: Load Data & Preprocessing (Production Grade)
import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def load_and_prep_data():
    logger.info("--- 1. Loading Files from Disk ---")
    
    try:
        df_listings = pd.read_csv('Listings_Data.csv')
        df_amenities = pd.read_csv('Amneties_Data.csv')
        df_engagement = pd.read_csv('User_Engagement_Data.csv')
        df_interactions = pd.read_csv('User_Interactions_Data.csv')
    except FileNotFoundError as e:
        logger.error(f"File missing: {e}")
        return None, None

    # --- A. SMART RENAMING (UPDATED) ---
    # CRITICAL FIX: Added 'bedrooms_int': 'bedrooms' mapping here
    rename_map = {
        'pp_price': 'price', 
        'listing_web_id': 'property_listing_id',
        'web_id': 'property_listing_id',
        'network_userid': 'user_id',
        'listing_entity_id': 'property_listing_id',
        'bedrooms_int': 'bedrooms',  # <--- THIS IS THE FIX
        'bathrooms_int': 'bathrooms' # Recommended if you have this column too
    }
    
    for df in [df_listings, df_amenities, df_engagement, df_interactions]:
        # Drop columns that might collide before renaming
        cols_to_drop = [v for k, v in rename_map.items() if k in df.columns and v in df.columns and k != v]
        if cols_to_drop:
            df.drop(columns=cols_to_drop, inplace=True)
        df.rename(columns=rename_map, inplace=True)

    # --- B. ID STANDARDIZATION ---
    id_cols = ['property_listing_id', 'user_id', 'agent_id', 'location_id']
    
    def clean_ids(df):
        for col in df.columns:
            if col in id_cols:
                df[col] = df[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

    for df in [df_listings, df_amenities, df_engagement, df_interactions]:
        clean_ids(df)

    # --- C. DEDUPLICATION (CRITICAL STEP) ---
    logger.info("--- 1.5 Dropping Duplicates ---")
    
    # 1. Deduplicate Listings
    initial_count = len(df_listings)
    if 'start_time' in df_listings.columns:
        df_listings = df_listings.sort_values('start_time', ascending=False)
        
    df_listings = df_listings.drop_duplicates(subset=['property_listing_id'], keep='first')
    logger.info(f"Listings Deduplicated: {initial_count} -> {len(df_listings)} (Dropped {initial_count - len(df_listings)})")

    # 2. Deduplicate Amenities
    df_amenities = df_amenities.drop_duplicates(subset=['property_listing_id', 'amenity_code'])

    # 3. Deduplicate Engagement
    df_engagement = df_engagement.drop_duplicates(subset=['property_listing_id'])

    # --- D. AMENITY AGGREGATION ---
    logger.info("--- 2. Building Amenity Strings ---")
    if not df_amenities.empty:
        amenity_group = df_amenities.groupby('property_listing_id')['amenity_code'].apply(
            lambda x: ','.join(sorted(set(x.dropna().astype(str))))
        ).reset_index()
        amenity_group.columns = ['property_listing_id', 'amenities']
    else:
        amenity_group = pd.DataFrame(columns=['property_listing_id', 'amenities'])

    # --- E. MASTER MERGE ---
    logger.info("--- 3. Merging Datasets ---")
    
    # 1. Merge Amenities
    df_listings = pd.merge(df_listings, amenity_group, on='property_listing_id', how='left')
    df_listings['amenities'] = df_listings['amenities'].fillna('')
    
    # 2. Merge Engagement
    engagement_cols = [c for c in df_engagement.columns if c not in df_listings.columns or c == 'property_listing_id']
    df_listings = pd.merge(df_listings, df_engagement[engagement_cols], on='property_listing_id', how='left')
    
    # Fill Engagement NaNs
    metric_cols = ['popularity_score', 'view_count', 'save_count']
    for col in metric_cols:
        if col in df_listings.columns:
            df_listings[col] = df_listings[col].fillna(0)
            
    logger.info(f" Data Loaded. Inventory Size: {len(df_listings)} Listings")
    return df_listings, df_interactions

# EXECUTE
df_listings, df_interactions = load_and_prep_data()

# --- VERIFICATION CHECK ---
print("\n--- DATA VERIFICATION ---")
if 'bedrooms' in df_listings.columns:
    print(f"✅ 'bedrooms' column exists.")
    print(f"Unique Bedroom Values: {df_listings['bedrooms'].unique()[:10]}")
else:
    print("❌ ERROR: 'bedrooms' column is MISSING!")

13:40:36 - INFO - --- 1. Loading Files from Disk ---
13:41:00 - INFO - --- 1.5 Dropping Duplicates ---
13:41:00 - INFO - Listings Deduplicated: 413900 -> 44127 (Dropped 369773)
13:41:02 - INFO - --- 2. Building Amenity Strings ---
13:43:02 - INFO - --- 3. Merging Datasets ---
13:43:03 - INFO -  Data Loaded. Inventory Size: 44127 Listings



--- DATA VERIFICATION ---
✅ 'bedrooms' column exists.
Unique Bedroom Values: [ 1.  2. nan  4.  3.  5.  6.  7.]


In [4]:
df_listings.columns

Index(['property_listing_id', 'location_id', 'latitude', 'longitude', 'location_name', 'full_location_path', 'property_title', 'property_address', 'listing_level', 'pending_verified_flag', 'listing_date', 'property_type', 'bedrooms', 'bedrooms_label', 'bathrooms', 'size_sqft', 'completion_status', 'furnished_flag', 'offering_type', 'category_id', 'property_type_id', 'price', 'price_period', 'quality_score', 'super_agent_score', 'amenities', 'view_count', 'save_count', 'lead_click_count', 'lead_submission_count', 'popularity_score'], dtype='object')

In [6]:
df_listings['bedrooms'].value_counts()

bedrooms
1.0    12374
2.0    10019
3.0     6446
4.0     4370
5.0     2398
6.0      658
7.0      225
Name: count, dtype: int64

In [7]:
df_listings['bedrooms_label'].value_counts()

bedrooms_label
1 Bed      12374
2 Bed      10019
3 Bed       6446
Studio      4655
4 Bed       4370
5 Bed       2398
6 Bed        658
7 Bed        161
7+ Beds       64
Name: count, dtype: int64

In [8]:
# Cell 3: Verification
if df_listings is not None:
    print("\n=== DATA AUDIT ===")
    print(f"1. Total Listings: {len(df_listings)}")
    
    # Check Critical Columns (Expanded for Safety)
    required = ['price', 'amenities', 'popularity_score', 'price_period', 'category_id']
    for col in required:
        if col in df_listings.columns:
            # Show a sample and the data type to be sure
            sample_val = df_listings[col].iloc[0]
            dtype = df_listings[col].dtype
            print(f"✅ Found '{col}' ({dtype}) - Sample: {sample_val}")
        else:
            print(f"❌ MISSING '{col}' - CRITICAL ERROR. Check Cell 2.")
            
    # Check Interactions
    print(f"\n2. Total Interactions: {len(df_interactions)}")
    if 'interaction_score' in df_interactions.columns:
        print(f"✅ Found 'interaction_score' - Sample: {df_interactions['interaction_score'].iloc[0]}")
    else:
        print("⚠️ Warning: 'interaction_score' missing (User Personalization might be limited)")


=== DATA AUDIT ===
1. Total Listings: 44127
✅ Found 'price' (float64) - Sample: 70000.0
✅ Found 'amenities' (object) - Sample: 
✅ Found 'popularity_score' (float64) - Sample: 0.0
✅ Found 'price_period' (object) - Sample: yearly
✅ Found 'category_id' (int64) - Sample: 2

2. Total Interactions: 1555000
✅ Found 'interaction_score' - Sample: 0


In [9]:
df_listings['offering_type'].value_counts()

offering_type
Residential for Sale    23720
Residential for Rent    17955
Commercial for Rent      1677
Commercial for Sale       775
Name: count, dtype: int64

In [10]:
# --- QC: Category vs Offering Type Alignment ---
import pandas as pd

def audit_category_mapping(df):
    print("=== 🧪 CATEGORY ALIGNMENT AUDIT ===")
    
    # 1. Create a Cross-tabulation
    # This shows us exactly which string labels are assigned to which numeric IDs
    alignment = pd.crosstab(
        df['offering_type'], 
        df['category_id'], 
        margins=True, 
        margins_name="Total"
    )
    
    print("\nMapping Matrix (Offering Type String vs. Category ID):")
    display(alignment)
    
    # 2. Check for "Hidden" Commercials
    # Look for Commercial strings that are NOT in Category 4
    hidden_comm = df[
        (df['offering_type'].str.contains('Commercial', case=False, na=False)) & 
        (df['category_id'] != 4)
    ]
    
    if not hidden_comm.empty:
        print(f"\n⚠️ WARNING: Found {len(hidden_comm)} Commercial listings NOT in Category 4.")
        print("Breakdown of these 'Hidden' Commercials:")
        print(hidden_comm['category_id'].value_counts())
        print("\nSample of misaligned rows:")
        display(hidden_comm[['offering_type', 'property_type', 'category_id']].head(10))
    else:
        print("\n✅ Clean Data: All 'Commercial' strings are correctly mapped to Category 4.")

# Execute the Audit
audit_category_mapping(df_listings)

=== 🧪 CATEGORY ALIGNMENT AUDIT ===

Mapping Matrix (Offering Type String vs. Category ID):


category_id,1,2,Total
offering_type,,,
Commercial for Rent,0,1677,1677
Commercial for Sale,775,0,775
Residential for Rent,0,17955,17955
Residential for Sale,23720,0,23720
Total,24495,19632,44127



⚠️ WARNING: Found 2452 Commercial listings NOT in Category 4.
Breakdown of these 'Hidden' Commercials:
category_id
2    1677
1     775
Name: count, dtype: int64

Sample of misaligned rows:


,offering_type,property_type,category_id
23,Commercial for Rent,Office Space,2
33,Commercial for Sale,Office Space,1
44,Commercial for Rent,Office Space,2
80,Commercial for Sale,Land,1
212,Commercial for Sale,Retail,1
225,Commercial for Rent,Office Space,2
230,Commercial for Sale,Land,1
294,Commercial for Sale,Office Space,1
316,Commercial for Sale,Office Space,1
332,Commercial for Sale,Office Space,1


In [12]:
# Cell 4: Advanced Feature Engineering & Enum Alignment (v12.4 FIXED)
import numpy as np
import pandas as pd
from datetime import datetime

def engineer_features_and_align_enums(df):
    logger.info("--- 4. Aligning Data with Proto Enums & Engineering Features ---")
    
    # 1. CATEGORY ALIGNMENT (Fixes 0 results for 3 & 4)
    category_map = {
        'Residential for Sale': 1,
        'Residential for Rent': 2,
        'Commercial for Sale': 3,
        'Commercial for Rent': 4
    }
    df['category_id'] = df['offering_type'].str.strip().map(category_map).fillna(0).astype(int)

    # 2. PROPERTY TYPE ALIGNMENT
    property_type_map = {
        'Apartment': 1, 'Villa': 35, 'Townhouse': 22, 'Penthouse': 20,
        'Duplex': 24, 'Hotel & Hotel Apartment': 45, 'Whole Building': 10,
        'Full Floor': 18, 'Residential Plot': 19, 'Office Space': 4,
        'Warehouse': 13, 'Show Room': 12, 'Shop': 21, 'Retail': 27,
        'Labor Camp': 11, 'Land': 19
    }
    for name, enum_id in property_type_map.items():
        df.loc[df['property_type'] == name, 'property_type_id'] = enum_id
    df['property_type_id'] = df['property_type_id'].fillna(0).astype(int)

    # 3. PRICE NORMALIZATION (Annual Rent vs Sale Price)
    df['is_sale'] = df['category_id'].isin([1, 3]).astype(int)
    
    multipliers = {'daily': 365, 'weekly': 52, 'monthly': 12, 'yearly': 1, 'sell': 1}
    df['price_period_clean'] = df['price_period'].astype(str).str.lower().str.strip()
    df['rent_multiplier'] = df['price_period_clean'].map(multipliers).fillna(1)

    df['feature_annual_rent'] = np.where(df['is_sale'] == 0, df['price'] * df['rent_multiplier'], 0.0)
    df['feature_sale_price'] = np.where(df['is_sale'] == 1, df['price'], 0.0)
    df['log_price'] = np.log1p(df['feature_annual_rent'] + df['feature_sale_price'])

    # 4. PHYSICAL ATTRIBUTES (Numeric Safety)
    df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce').fillna(0).astype(int)
    df['bathrooms'] = df['bathrooms'].astype(str).str.extract('(\d+)').fillna(0).astype(int)

    # 5. ENUM STRING-TO-INT MAPPING (CRITICAL FIX FOR CRASH)
    # Mapping Completion Status strings to Enum values
    comp_map = {'completed': 0, 'off_plan': 1, 'off-plan': 1}
    df['completion_status'] = df['completion_status'].astype(str).str.lower().str.strip().map(comp_map).fillna(0).astype(int)

    # Mapping Furnished strings (YES=1, NO=2, PARTLY=3, ALL/Unknown=0)
    furn_map = {'yes': 1, 'no': 2, 'partly': 3, 'partial': 3, 'furnished': 1, 'unfurnished': 2}
    if 'furnished_flag' in df.columns:
        df['furnished_flag'] = df['furnished_flag'].astype(str).str.lower().str.strip().map(furn_map).fillna(0).astype(int)

    # 6. QUALITY & FRESHNESS
    level_map = {'premium': 2, 'featured': 1, 'basic': 0}
    df['listing_level_score'] = df['listing_level'].str.lower().map(level_map).fillna(0)
    
    df['listing_date'] = pd.to_datetime(df['listing_date'], errors='coerce')
    df['days_active'] = (pd.Timestamp.now() - df['listing_date']).dt.days.fillna(30).clip(lower=0)

    logger.info("--- 🏁 Logic Alignment Complete ---")
    return df

# Execute
df_listings = engineer_features_and_align_enums(df_listings)

# Verify counts again to be 100% sure
print(f"✅ Commercial Rent (Cat 4): {len(df_listings[df_listings['category_id']==4])}")
print(f"✅ Completion Status 0 (Completed): {len(df_listings[df_listings['completion_status']==0])}")

13:46:57 - INFO - --- 4. Aligning Data with Proto Enums & Engineering Features ---
13:46:58 - INFO - --- 🏁 Logic Alignment Complete ---


✅ Commercial Rent (Cat 4): 1677
✅ Completion Status 0 (Completed): 38229


In [14]:
# Cell 3.5: Find the Missing Price Column
import pandas as pd

# 1. Print ALL columns to spot the right one
print("--- ALL COLUMNS ---")
print(df_listings.columns.tolist())

# 2. Smart Search for "Price"
print("\n--- CANDIDATE PRICE COLUMNS ---")
candidates = [c for c in df_listings.columns if 'price' in c.lower()]
print(candidates)

# 3. Check the first row of these candidates to confirm
if candidates:
    print("\n--- SAMPLE VALUES ---")
    print(df_listings[candidates].head(1))

--- ALL COLUMNS ---
['property_listing_id', 'location_id', 'latitude', 'longitude', 'location_name', 'full_location_path', 'property_title', 'property_address', 'listing_level', 'pending_verified_flag', 'listing_date', 'property_type', 'bedrooms', 'bedrooms_label', 'bathrooms', 'size_sqft', 'completion_status', 'furnished_flag', 'offering_type', 'category_id', 'property_type_id', 'price', 'price_period', 'quality_score', 'super_agent_score', 'amenities', 'view_count', 'save_count', 'lead_click_count', 'lead_submission_count', 'popularity_score', 'is_sale', 'price_period_clean', 'rent_multiplier', 'feature_annual_rent', 'feature_sale_price', 'log_price', 'listing_level_score', 'days_active']

--- CANDIDATE PRICE COLUMNS ---
['price', 'price_period', 'price_period_clean', 'feature_sale_price', 'log_price']

--- SAMPLE VALUES ---
     price price_period price_period_clean  feature_sale_price  log_price
0  70000.0       yearly             yearly                 0.0  11.156265


In [15]:
# Cell 3: Feature Engineering (Merged: Pricing + Bedrooms + Cleaning)
import pandas as pd
import numpy as np
import re
import logging
from datetime import datetime

logger = logging.getLogger(__name__)

def engineer_features(df):
    logger.info("--- 3. Feature Engineering (World-Class Logic) ---")
    
    # =========================================================
    # PART A: PROPERTY TYPE ID (Required for Bedroom Logic)
    # =========================================================
    # We do this first so we know if a property is Commercial or Residential
    type_map = {
        'apartment': 1, 'flat': 1, 'villa': 35, 'house': 35, 'townhouse': 22, 
        'penthouse': 20, 'duplex': 24, 'office': 4, 'warehouse': 13, 
        'shop': 21, 'retail': 27, 'whole building': 10, 'land': 19,
        'bunker': 13, 'showroom': 12, 'commercial': 4
    }
    
    if 'property_type_id' not in df.columns: 
        df['property_type_id'] = 0
        
    # Map from string if ID is missing (0)
    mask_type = (df['property_type_id'] == 0)
    if 'property_type' in df.columns:
        df.loc[mask_type, 'property_type_id'] = df.loc[mask_type, 'property_type'].astype(str).str.lower().map(type_map).fillna(0)
    
    # =========================================================
    # PART B: THE BEDROOM FIX (CRITICAL)
    # =========================================================
    logger.info("... Applying Bedroom Logic ...")

    # Commercial/Land IDs (Where bedrooms should ALWAYS be 0)
    COMMERCIAL_IDS = [4, 5, 10, 11, 12, 13, 19, 21, 27, 29, 34, 42, 44]

    def smart_bedroom_logic(row):
        # Inputs
        label = str(row.get('bedrooms_label', '')).lower().strip()
        current_val = row.get('bedrooms', np.nan) 
        prop_type = row.get('property_type_id', 0)

        # RULE 1: Commercial/Land is ALWAYS 0
        if prop_type in COMMERCIAL_IDS:
            return 0.0

        # RULE 2: Explicit "Studio" label -> 0
        if 'studio' in label:
            return 0.0

        # RULE 3: "7+" label -> 7
        if '7+' in label:
            return 7.0

        # RULE 4: Extract number from label (e.g. "2 Beds" -> 2)
        match = re.search(r'(\d+)', label)
        if match:
            return float(match.group(1))

        # RULE 5: Fallback to existing integer (if valid)
        if pd.notna(current_val) and current_val >= 0:
            return float(current_val)

        return 0.0

    # Apply the logic
    # Ensure 'bedrooms' is initialized from 'bedrooms_int' if not done in Cell 2
    if 'bedrooms_int' in df.columns and 'bedrooms' not in df.columns:
        df['bedrooms'] = df['bedrooms_int']
        
    df['bedrooms'] = df.apply(smart_bedroom_logic, axis=1)

    # =========================================================
    # PART C: SMART PRICING & MARKET SEGMENTS
    # =========================================================
    logger.info("... Applying Pricing Logic ...")

    # 1. Define Category (Buy vs Rent)
    cat_map = {
        'residential for sale': 1, 'buy': 1, 'sale': 1,
        'residential for rent': 2, 'rent': 2,
        'commercial for sale': 3, 'commercial for rent': 4
    }
    if 'category_id' not in df.columns:
        if 'offering_type' in df.columns:
             df['category_id'] = df['offering_type'].astype(str).str.lower().map(cat_map).fillna(0).astype(int)
        else:
             df['category_id'] = 0

    # Explicit Boolean
    df['is_sale'] = df['category_id'].isin([1, 3]).astype(int)

    # 2. Handle Rental Periods
    if 'price_period' not in df.columns: df['price_period'] = 'unknown'
    df['price_period'] = df['price_period'].astype(str).str.lower().str.strip()
    
    rent_multipliers = {
        'daily': 365, 'weekly': 52, 'monthly': 12, 
        'yearly': 1, 'sell': 1, 'unknown': 1, 'nan': 1, '': 1
    }
    df['rent_multiplier'] = df['price_period'].map(rent_multipliers).fillna(1)

    # 3. Split Pricing Features
    df['feature_annual_rent'] = 0.0
    df['feature_sale_price'] = 0.0
    
    mask_rent = (df['is_sale'] == 0)
    mask_sale = (df['is_sale'] == 1)
    
    # Safely calculate (fillNa with 0 before multiplying to avoid errors)
    df.loc[mask_rent, 'feature_annual_rent'] = df.loc[mask_rent, 'price'].fillna(0) * df.loc[mask_rent, 'rent_multiplier']
    df.loc[mask_sale, 'feature_sale_price'] = df.loc[mask_sale, 'price'].fillna(0)

    # 4. Log Price
    df['log_price'] = np.log1p(df['feature_annual_rent'] + df['feature_sale_price'])

    # 5. Price Per Sqft
    df['price_per_sqft'] = 0.0
    safe_size = df['size_sqft'].replace(0, 1)
    
    df.loc[mask_rent, 'price_per_sqft'] = df.loc[mask_rent, 'feature_annual_rent'] / safe_size.loc[mask_rent]
    df.loc[mask_sale, 'price_per_sqft'] = df.loc[mask_sale, 'feature_sale_price'] / safe_size.loc[mask_sale]

    # =========================================================
    # PART D: ML SIGNALS & CLEANING
    # =========================================================
    
    # 6. Freshness (Days Active)
    date_col = 'start_time' if 'start_time' in df.columns else 'listing_date'
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce').fillna(pd.Timestamp.now())
        df['days_active'] = (pd.Timestamp.now() - df[date_col]).dt.days.clip(lower=0)
    else:
        df['days_active'] = 0

    # 7. Listing Level
    if 'listing_level' in df.columns:
        level_map = {'standard': 1, 'premium': 2, 'featured': 3}
        df['listing_level_score'] = df['listing_level'].astype(str).str.lower().map(level_map).fillna(1).astype(int)
    else:
        df['listing_level_score'] = 1

    # 8. Completion Status
    def clean_completion(val):
        s = str(val).lower()
        if 'complete' in s or 'ready' in s: return 0
        if 'off' in s or 'plan' in s: return 1
        return 0
        
    if 'completion_status' in df.columns:
        df['completion_status'] = df['completion_status'].apply(clean_completion)
    else:
        df['completion_status'] = 0

    # 9. Furnished
    def clean_furnished(val):
        s = str(val).lower()
        if 'yes' in s or ('furnished' in s and 'un' not in s): return 1
        if 'no' in s or 'unfurnished' in s: return 2
        return 2 # Default to No/Partly
        
    if 'furnished' in df.columns:
        df['furnished_flag'] = df['furnished'].apply(clean_furnished)
    else:
        df['furnished_flag'] = 0

    # 10. Super Agent
    if 'is_super_agent' in df.columns:
        df['super_agent_score'] = df['is_super_agent'].astype(str).str.lower().map({'t': 1, 'true': 1, '1': 1}).fillna(0).astype(int)
    else:
        df['super_agent_score'] = 0
        
    # 11. Text Cleaning
    for col in ['images', 'video_url', 'view_360', 'agent_name', 'broker_name']:
        if col not in df.columns: df[col] = ""
        else: df[col] = df[col].astype(str).str.replace('nan', '', case=False, regex=False)

    if 'popularity_score' in df.columns:
        df['popularity_score'] = pd.to_numeric(df['popularity_score'], errors='coerce').fillna(0)

    logger.info(f"✅ Features Engineered. Inventory Size: {len(df)}")
    
    # --- Final Audit ---
    print("\n--- AUDIT: BEDROOMS ---")
    print(df['bedrooms'].value_counts().sort_index().head(8))
    print("\n--- AUDIT: PRICING ---")
    print(df[['category_id', 'price', 'is_sale', 'feature_annual_rent']].head(5))

    return df

# Execute
df_listings = engineer_features(df_listings)

14:36:27 - INFO - --- 3. Feature Engineering (World-Class Logic) ---
14:36:27 - INFO - ... Applying Bedroom Logic ...
14:36:27 - INFO - ... Applying Pricing Logic ...
14:36:27 - INFO - ✅ Features Engineered. Inventory Size: 44127



--- AUDIT: BEDROOMS ---
bedrooms
0.0     7655
1.0    12370
2.0    10017
3.0     6443
4.0     4370
5.0     2398
6.0      658
7.0      216
Name: count, dtype: int64

--- AUDIT: PRICING ---
   category_id      price  is_sale  feature_annual_rent
0            2    70000.0        0              70000.0
1            2   150000.0        0             150000.0
2            1  1950000.0        1                  0.0
3            1   599000.0        1                  0.0
4            1  2449900.0        1                  0.0


In [16]:
# Cell 4: Train Production Model (Smart Features & Artifact Saving)
import xgboost as xgb
import pandas as pd
import joblib
import os

def train_production_model(df_listings, df_interactions):
    logger.info("--- 4. Training Production Model ---")
    
    # 1. Merge Data (The "Learning Set")
    # We train on actual user behavior (What did they click?)
    # Inner merge: Only train on listings that have history
    df_train = pd.merge(df_interactions, df_listings, on='property_listing_id', how='inner')
    
    # QC: Ensure we have data
    if len(df_train) < 50:
        logger.warning(f"⚠️ Low Interaction Data ({len(df_train)} rows). Model might be weak.")
        # Fallback: If interactions are missing, train on "Popularity" (Views)
        if len(df_train) < 10:
             logger.info("⚠️ Switching to Popularity-based training (Cold Start)")
             df_train = df_listings.copy()
             df_train['interaction_score'] = df_train['popularity_score'] # Target is now popularity

    logger.info(f"Training on {len(df_train)} examples.")

    # 2. DEFINE FEATURES (The "Smart Signals")
    # CRITICAL: We use the engineered features, NOT the raw 'price'
    MODEL_FEATURES = [
        # --- Value Signals ---
        'feature_annual_rent',   # Smart Rent (0 for Sales)
        'feature_sale_price',    # Smart Sale (0 for Rents)
        'is_sale',               # The Switch
        'price_per_sqft',        # Normalized Value
        
        # --- Trust & Quality ---
        'listing_level_score',   # Paid/Premium
        'super_agent_score',     # Trust
        'days_active',           # Freshness
        'popularity_score',      # Social Proof
        
        # --- Specs ---
        'size_sqft', 
        'bedrooms', 
        'bathrooms',
        'completion_status',     # Ready vs Off-plan
        'furnished_flag',        # Furnished
        'property_type_id',      # Villa/Apt
        'category_id'            # Context
    ]

    # 3. TYPE SAFETY ENFORCEMENT
    # XGBoost crashes on strings. Force everything to float.
    for col in MODEL_FEATURES:
        if col not in df_train.columns:
            logger.warning(f"Feature '{col}' missing. Creating with 0.")
            df_train[col] = 0.0
        else:
            df_train[col] = pd.to_numeric(df_train[col], errors='coerce').fillna(0)

    # 4. Prepare Matrices
    X = df_train[MODEL_FEATURES]
    y = df_train['interaction_score'] # Target: 1 (View) to 50 (Lead)

    # 5. Initialize XGBoost Regressor
    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=200,        # More trees for better accuracy
        learning_rate=0.03,      # Slower learning = more robust
        max_depth=7,             # Deeper trees to capture (Price vs Location) nuances
        subsample=0.8,           # Prevent overfitting
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42
    )
    
    # 6. Train
    model.fit(X, y)
    
    # 7. INTELLIGENCE REPORT
    importance = pd.DataFrame({
        'Feature': MODEL_FEATURES,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\n=== 🧠 MODEL INTELLIGENCE REPORT ===")
    print(importance.head(10))
    
    # 8. SAVE ARTIFACTS (CRITICAL FOR API)
    # The API needs the model AND the list of features expected
    artifacts = {
        'model': model,
        'features': MODEL_FEATURES,
        'version': 'v12.10_Pro'
    }
    
    os.makedirs('artifacts', exist_ok=True)
    joblib.dump(artifacts, 'artifacts/ranker_model.pkl')
    
    # Save the processed inventory so API doesn't need to re-engineer features
    # Drop large text columns to save space if needed, but keep 'amenities'
    df_listings.to_parquet('artifacts/inventory.parquet', index=False)
    
    logger.info("✅ Model & Inventory saved to 'artifacts/'")
    
    return model, MODEL_FEATURES

# Execute
ranker_model, MODEL_FEATURES = train_production_model(df_listings, df_interactions)

14:37:02 - INFO - --- 4. Training Production Model ---
14:37:03 - INFO - Training on 250 examples.



=== 🧠 MODEL INTELLIGENCE REPORT ===
                Feature  Importance
8             size_sqft    0.164325
4   listing_level_score    0.162654
7      popularity_score    0.104512
0   feature_annual_rent    0.094022
3        price_per_sqft    0.093633
1    feature_sale_price    0.086156
6           days_active    0.078980
10            bathrooms    0.069223
13     property_type_id    0.059047
9              bedrooms    0.058325


14:37:03 - INFO - ✅ Model & Inventory saved to 'artifacts/'


In [32]:
# Cell 5: Save Production Artifacts (Strict Type Enforcement & Feature Alignment)
import shutil
import os
import joblib
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

def save_artifacts(model, features, df_listings, df_interactions):
    logger.info("--- 5. Saving Artifacts ---")
    
    # 1. Define the Schema (API Response + Model Features)
    # We MUST include the engineered features so the API doesn't have to recalculate them
    api_required_columns = [
        # --- Identity ---
        'property_listing_id', 'location_id', 'agent_id', 'broker_id',
        
        # --- Display Data ---
        'property_title', 'property_address', 'latitude', 'longitude',
        'price', 'price_period', 'currency', 
        'images', 'video_url', 'view_360',
        'agent_name', 'broker_name', 'is_super_agent',
        'verified', 'listing_level', 'location_name', 'full_location_path',
        
        # --- Specs ---
        'bedrooms', 'bathrooms', 'size_sqft', 'amenities',
        'category_id', 'property_type_id', 'property_type',
        'completion_status', 'furnished_flag',
        
        # --- Smart Features (CRITICAL FOR MODEL) ---
        'feature_annual_rent', 
        'feature_sale_price', 
        'is_sale',
        'listing_level_score', 
        'super_agent_score', 
        'popularity_score', 
        'price_per_sqft', 
        'days_active'
    ]
    
    final_df = df_listings.copy()
    
    # 2. FILL MISSING COLUMNS
    # Ensure every required column exists, even if empty
    for col in api_required_columns:
        if col not in final_df.columns:
            if col in ['price', 'latitude', 'longitude', 'size_sqft', 'bedrooms', 'bathrooms']:
                final_df[col] = 0.0
            elif col in ['images', 'amenities']:
                final_df[col] = ""
            elif col in ['feature_annual_rent', 'feature_sale_price']:
                final_df[col] = 0.0 # Critical safety net
            else:
                final_df[col] = None

    # 3. FORCE NUMERIC TYPES (The Fix)
    # This ensures Parquet saves them as numbers, not strings.
    numeric_cols = [
        'price', 'bedrooms', 'bathrooms', 'size_sqft', 
        'latitude', 'longitude', 
        'category_id', 'property_type_id', 
        'listing_level_score', 'super_agent_score', 'popularity_score', 
        'price_per_sqft', 'days_active',
        'completion_status', 'furnished_flag',
        'feature_annual_rent', 'feature_sale_price', 'is_sale'
    ]
    
    logger.info("Enforcing numeric types for Parquet...")
    for col in numeric_cols:
        if col in final_df.columns:
            final_df[col] = pd.to_numeric(final_df[col], errors='coerce').fillna(0)

    # 4. STRING CLEANUP
    final_df['property_listing_id'] = final_df['property_listing_id'].astype(str)
    final_df['location_id'] = final_df['location_id'].fillna('0').astype(str)
    final_df['agent_id'] = final_df['agent_id'].fillna('0').astype(str)
    
    # 5. NLP VOCAB (For future Content-Based Filtering)
    logger.info("Generating NLP Vocabulary...")
    vectorizer = CountVectorizer(tokenizer=lambda x: x.split(','), token_pattern=None)
    vectorizer.fit(final_df['amenities'].fillna(''))
    vocab = vectorizer.vocabulary_
    
    # 6. SAVE
    os.makedirs("artifacts", exist_ok=True)
    
    brain = {
        "version": "12.1_Pro",
        "model": model,
        "features": features, # The list of features the model expects
        "vocab": vocab
    }
    
    # Save Model
    joblib.dump(brain, f"artifacts/brain.pkl", compress=3)
    
    # Save Inventory (Only the columns we need)
    # Filter final_df to only include columns that actually exist in api_required_columns
    cols_to_save = [c for c in api_required_columns if c in final_df.columns]
    final_df[cols_to_save].to_parquet(f"artifacts/inventory.parquet", index=False)
    
    # Save User Data
    df_interactions.to_parquet(f"artifacts/user_data.parquet", index=False)
    
    logger.info(f"✅ SUCCESS: Artifacts saved (Types Enforced).")
    logger.info(f"   - Brain: artifacts/brain.pkl")
    logger.info(f"   - Inventory: artifacts/inventory.parquet ({len(final_df)} listings)")

# Execute
save_artifacts(ranker_model, MODEL_FEATURES, df_listings, df_interactions)

15:13:46 - INFO - --- 5. Saving Artifacts ---
15:13:47 - INFO - Enforcing numeric types for Parquet...
15:13:47 - INFO - Generating NLP Vocabulary...
15:13:50 - INFO - ✅ SUCCESS: Artifacts saved (Types Enforced).
15:13:50 - INFO -    - Brain: artifacts/brain.pkl
15:13:50 - INFO -    - Inventory: artifacts/inventory.parquet (44127 listings)


In [22]:
# Cell 6: Verify Artifacts (Final System Check)
import joblib
import pandas as pd
import numpy as np

def verify_artifacts():
    print("--- 6. Running Final Sanity Check ---")
    
    # 1. LOAD ARTIFACTS
    try:
        # Load the "Brain" (Model + Metadata)
        brain = joblib.load("artifacts/brain.pkl")
        model = brain['model']
        features = brain['features']
        version = brain.get('version', 'Unknown')
        
        # Load the "Inventory" (Data)
        inventory = pd.read_parquet("artifacts/inventory.parquet")
        
        print(f"✅ Loaded Brain v{version}")
        print(f"✅ Loaded Inventory: {len(inventory)} listings")
        print(f"✅ Model expects {len(features)} features")
        
    except Exception as e:
        print(f"❌ CRITICAL: Failed to load artifacts. {e}")
        return

    # 2. SIMULATE PREDICTION (The "Real-World" Test)
    print("\n[C] Simulating AI Prediction on Random Listing:")
    
    # Pick a random sample
    sample_df = inventory.sample(1).copy()
    
    # Prepare Input Features (Ensure exact match with training)
    X_input = pd.DataFrame()
    for col in features:
        if col in sample_df.columns:
            # Ensure numeric safety
            X_input[col] = pd.to_numeric(sample_df[col], errors='coerce').fillna(0)
        else:
            print(f"⚠️ Warning: Feature '{col}' missing in inventory. Defaulting to 0.")
            X_input[col] = 0.0

    # 3. PREDICT
    try:
        # Run the model
        score = model.predict(X_input)[0]
        
        # 4. REPORT
        row = sample_df.iloc[0]
        is_sale = row.get('is_sale', 0)
        mode = "SALE" if is_sale == 1 else "RENT"
        
        print(f"   - Listing ID: {row['property_listing_id']}")
        print(f"   - Context: {mode} (Category {row.get('category_id')})")
        print(f"   - Raw Price: {row.get('price', 0):,.0f} ({row.get('price_period', 'N/A')})")
        
        # The Critical Check: Did the Smart Features save correctly?
        print(f"   - 🧠 Smart Features: AnnualRent={row.get('feature_annual_rent', 0):,.0f}, SalePrice={row.get('feature_sale_price', 0):,.0f}")
        
        print(f"   - 🚀 PREDICTED SCORE: {score:.4f}")
        
        if score > 0:
            print("\n✅ SYSTEM READY FOR DEPLOYMENT.")
        else:
            print("\n⚠️ SYSTEM READY, but score is 0 (Check if features are empty)")
            
    except Exception as e:
        print(f"\n❌ PREDICTION FAILED: {e}")
        print(f"   - Input Shapes: {X_input.shape}")

# Execute
verify_artifacts()

--- 6. Running Final Sanity Check ---
✅ Loaded Brain v12.1_Pro
✅ Loaded Inventory: 44127 listings
✅ Model expects 15 features

[C] Simulating AI Prediction on Random Listing:
   - Listing ID: 14576776
   - Context: RENT (Category 4)
   - Raw Price: 400,000 (yearly)
   - 🧠 Smart Features: AnnualRent=400,000, SalePrice=0
   - 🚀 PREDICTED SCORE: 0.0242

✅ SYSTEM READY FOR DEPLOYMENT.


In [23]:
import pandas as pd

# --- 1. LOAD & FIX DATA ---
# Load the data again to be sure (or use your existing df_raw)
df_raw = pd.read_csv('Listings_Data.csv')

# Define rename map
rename_map = {
    'pp_price': 'price', 
    'listing_web_id': 'property_listing_id',
    'web_id': 'property_listing_id',
    'listing_entity_id': 'property_listing_id',
    'offering_type': 'category_id' 
}

# [FIX] Check if target columns exist before renaming to avoid duplicates
# If 'category_id' already exists in the dataframe, drop the old one before renaming 'offering_type'
if 'category_id' in df_raw.columns and 'offering_type' in df_raw.columns:
    df_raw.drop(columns=['category_id'], inplace=True)

df_raw.rename(columns=rename_map, inplace=True)

# [FIX] Final Safety Net: Remove any duplicate columns that might still exist
df_raw = df_raw.loc[:, ~df_raw.columns.duplicated()]

# Ensure numeric
df_raw['price'] = pd.to_numeric(df_raw['price'], errors='coerce').fillna(0)

# --- 2. CONTEXT AWARE INSPECTION ---
if 'price_period' not in df_raw.columns:
    df_raw['price_period'] = 'unknown'

if 'category_id' not in df_raw.columns:
    print("⚠️ Error: 'category_id' column is missing. Check your CSV column names.")
    # Create dummy to prevent crash
    df_raw['category_id'] = 0 

print("📊 PRICE DISTRIBUTION REPORT\n")

# Grouping Logic (Should work now)
groups = df_raw.groupby(['category_id', 'price_period'])

# Calculate stats
stats = groups['price'].agg(
    Count='count',
    Min='min',
    Median='median',
    Max='max'
).reset_index().sort_values('category_id')

# Display the Stats Table
print("--- Price Ranges by Category & Period ---")
display(stats.style.format({
    'Min': '{:,.0f}', 
    'Median': '{:,.0f}', 
    'Max': '{:,.0f}'
}))

# --- 3. SPOT CHECK: EXTREME LOWS ---
print("\n--- 🔍 INSPECTING THE LOWEST PRICES (< 1,000) ---")
low_price_samples = df_raw[df_raw['price'] > 0].nsmallest(10, 'price')

if not low_price_samples.empty:
    cols_to_show = ['property_listing_id', 'category_id', 'price_period', 'price']
    if 'title' in df_raw.columns: cols_to_show.append('title')
    display(low_price_samples[[c for c in cols_to_show if c in df_raw.columns]])
else:
    print("No non-zero low prices found.")

📊 PRICE DISTRIBUTION REPORT

--- Price Ranges by Category & Period ---


,category_id,price_period,Count,Min,Median,Max
0,Commercial for Rent,daily,28,90,90,90
1,Commercial for Rent,monthly,212,"1,200","4,500","70,000"
2,Commercial for Rent,yearly,13534,"5,000","250,958","26,500,000"
3,Commercial for Sale,sell,6484,"100,000","4,000,000","300,000,000"
4,Residential for Rent,daily,1204,199,505,"139,990"
5,Residential for Rent,monthly,7221,"1,417","8,999","600,000"
6,Residential for Rent,weekly,884,"1,199","3,800","165,000"
7,Residential for Rent,yearly,147325,"10,000","160,000","60,000,000"
8,Residential for Sale,sell,237008,"100,000","2,398,971","500,000,000"



--- 🔍 INSPECTING THE LOWEST PRICES (< 1,000) ---


,property_listing_id,category_id,price_period,price
103699,11998010,Commercial for Rent,daily,90.0
103700,11998010,Commercial for Rent,daily,90.0
103701,11998010,Commercial for Rent,daily,90.0
103702,11998010,Commercial for Rent,daily,90.0
103703,11998010,Commercial for Rent,daily,90.0
103704,11998010,Commercial for Rent,daily,90.0
103705,11998010,Commercial for Rent,daily,90.0
103706,11998010,Commercial for Rent,daily,90.0
103707,11998010,Commercial for Rent,daily,90.0
103708,11998010,Commercial for Rent,daily,90.0


In [24]:
import pandas as pd

# 1. Load the Inventory
df = pd.read_parquet("artifacts/inventory.parquet")

# 2. Check what unique bedroom numbers actually exist in the DB
print("--- Unique Bedroom Counts in Database ---")
print(sorted(df['bedrooms'].unique()))

# 3. Filter for 2 or 3 Bedrooms
target_beds = [2, 3]
mask_beds = df['bedrooms'].isin(target_beds)
df_filtered = df[mask_beds]

print(f"\n--- Total Properties with 2 or 3 Beds: {len(df_filtered)} ---")

# 4. Deep Dive: Check by Category (Rent vs Sale)
# category_id: 1 = Sale, 2 = Rent
print("\n--- Breakdown by Category ---")
print(df_filtered['category_id'].value_counts())

# 5. Preview the Data (if any exist)
if not df_filtered.empty:
    print("\n--- Preview of Data ---")
    cols_to_show = ['property_listing_id', 'property_title', 'bedrooms', 'category_id', 'price']
    print(df_filtered[cols_to_show].head(10))
else:
    print("\n❌ No properties found with 2 or 3 bedrooms in the DataFrame.")

--- Unique Bedroom Counts in Database ---
[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0)]

--- Total Properties with 2 or 3 Beds: 16460 ---

--- Breakdown by Category ---
category_id
1    9140
2    7319
4       1
Name: count, dtype: int64

--- Preview of Data ---
   property_listing_id                                     property_title  bedrooms  category_id      price
1             14917393                 Marina View | Unfurnished | Vacant       2.0            2   150000.0
4             14803150       PRIVATE BEACH ACCESS | POOL VIEW | BEST DEAL       2.0            1  2449900.0
5             14675237  Full Sea View| Access to Beach Club|  Unfurnished       2.0            2   250000.0
12            14925053  Luxurious 2-BR | Sea View | All Amenities | Gym |       2.0            2    75000.0
14            13825157                  Exclusive | Upgraded | High Floor       3.0            1  3200000.0

In [25]:
# --- QC: Global Market Mapping ---
import pandas as pd

def check_all_listings_integrity(df):
    print("=== 📊 GLOBAL DATA DISTRIBUTION CHECK ===")
    
    # 1. Total Counts by Category
    # 1=Res Sale, 2=Res Rent, 3=Comm Sale, 4=Comm Rent
    print("\nListing Counts by Category ID:")
    category_summary = df['category_id'].value_counts().sort_index()
    print(category_summary)

    # 2. Detailed Cross-Section (Category vs Property Type)
    # This shows us exactly which types are in which categories
    print("\nMatrix: Property Type vs Category (Top 20):")
    cross_tab = pd.crosstab(df['property_type_id'], df['category_id'])
    print(cross_tab.head(20))

    # 3. Commercial Rent Deep Dive (Category 4)
    comm_rent = df[df['category_id'] == 4]
    if not comm_rent.empty:
        print("\n--- Commercial Rent Details (Category 4) ---")
        print(f"Total Found: {len(comm_rent)}")
        print("\nProperty Types in Category 4:")
        print(comm_rent['property_type'].value_counts())
        
        print("\nPrice Periods in Category 4:")
        print(comm_rent['price_period'].value_counts(dropna=False))
        
        print("\nBedroom values in Category 4 (Should be 0.0):")
        print(comm_rent['bedrooms'].value_counts())
    else:
        print("\n❌ ALERT: Category 4 (Commercial Rent) is COMPLETELY EMPTY.")
        # Check if they are accidentally in Category 2
        potential_comm = df[(df['category_id'] == 2) & (df['property_type_id'].isin([4, 13, 21, 27]))]
        if not potential_comm.empty:
            print(f"Found {len(potential_comm)} Commercial-type properties incorrectly assigned to Category 2 (Residential Rent).")

    # 4. Check for 'NaN' or Invalid IDs
    nan_cats = df['category_id'].isna().sum()
    nan_types = df['property_type_id'].isna().sum()
    print(f"\nMissing Data: {nan_cats} Null Categories, {nan_types} Null Property Types")

# Execute
check_all_listings_integrity(df_listings)

=== 📊 GLOBAL DATA DISTRIBUTION CHECK ===

Listing Counts by Category ID:
category_id
1    23720
2    17955
3      775
4     1677
Name: count, dtype: int64

Matrix: Property Type vs Category (Top 20):
category_id           1      2    3     4
property_type_id                         
0                    12      7   19    62
1                 16986  13031    0     1
4                     1      0  296  1040
10                   17      3   22     8
11                    0      0   14    15
12                    0      0    1    16
13                    0      0   51   253
18                    2      0    8    16
19                  490      0  191    33
20                  175     88    0     0
21                    0      0   70   126
22                 1851   1442    0     0
24                   96     54    0     0
27                    0      0   99    78
35                 4004   3171    4    29
45                   86    159    0     0

--- Commercial Rent Details (Category 4) --

In [26]:
# --- QC: Audit Raw Listing Content ---
import pandas as pd

def audit_listings_content(df):
    print("=== 🔍 RAW LISTINGS AUDIT: COMMERCIAL TYPES IN CATEGORY 2 ===\n")
    
    # Target property type IDs we found: 13 (Warehouse), 21 (Shop), 27 (Retail)
    commercial_ids = [13, 21, 27, 4, 12]
    
    # Sample properties that SHOULD be Category 4 but are currently Category 2
    audit_df = df[(df['category_id'] == 2) & (df['property_type_id'].isin(commercial_ids))].copy()
    
    if audit_df.empty:
        print("No commercial properties found in Category 2.")
        return

    # Select columns to inspect raw data
    # We want to see what the agent actually wrote
    cols_to_view = [
        'property_listing_id', 
        'property_type', 
        'property_type_id',
        'category_id',
        'offering_type',  # The original string from CSV
        'property_title', 
        'price',
        'price_period'
    ]
    
    # Clean selection for display
    display_cols = [c for c in cols_to_view if c in audit_df.columns]
    
    print(f"Showing sample of {len(audit_df)} commercial properties currently labeled as 'Rent' (Category 2):")
    print(audit_df[display_cols].head(15).to_string(index=False))

    print("\n--- Summary of 'offering_type' strings for these properties ---")
    if 'offering_type' in audit_df.columns:
        print(audit_df['offering_type'].value_counts())
    else:
        print("Column 'offering_type' not found in dataframe.")

# Execute Audit
audit_listings_content(df_listings)

=== 🔍 RAW LISTINGS AUDIT: COMMERCIAL TYPES IN CATEGORY 2 ===

No commercial properties found in Category 2.


In [20]:
# --- MASTER DATA QUALITY CONTROL & AUDIT ---
import pandas as pd
import numpy as np

def perform_master_qc(df):
    print("==================================================")
    print("📈 WORLD-CLASS DATA AUDIT REPORT")
    print("==================================================\n")
    
    # 1. General Info
    print(f"Total Listings: {len(df)}")
    print(f"Total Columns: {len(df.columns)}")
    
    # 2. Detailed Column Analysis
    analysis = []
    for col in df.columns:
        null_count = df[col].isna().sum()
        null_pct = (null_count / len(df)) * 100
        dtype = df[col].dtype
        unique_count = df[col].nunique()
        
        analysis.append({
            'Column': col,
            'Dtype': dtype,
            'Unique': unique_count,
            'Nulls': null_count,
            'Null %': f"{null_pct:.1f}%"
        })
    
    analysis_df = pd.DataFrame(analysis)
    print("--- 1. COLUMN SUMMARY ---")
    print(analysis_df.to_string(index=False))
    
    # 3. Targeted Values Audit (The "Label Hunt")
    print("\n--- 2. DETAILED VALUE DISTRIBUTION (Top 5 for Key Columns) ---")
    key_cols = [
        'category_id', 'property_type_id', 'offering_type', 
        'property_type', 'price_period', 'bedrooms', 'bathrooms'
    ]
    
    for col in key_cols:
        if col in df.columns:
            print(f"\nDistribution for [{col}]:")
            # Show top 5 unique values and their raw counts
            counts = df[col].value_counts(dropna=False).head(10)
            for val, count in counts.items():
                print(f"  - '{val}' (Type: {type(val).__name__}): {count}")
        else:
            print(f"\n⚠️ Column [{col}] is missing from the table!")

    # 4. Consistency Check: The "Misalignment" Detector
    print("\n--- 3. CONSISTENCY & LOGIC AUDIT ---")
    
    # Check if 'bedrooms' has strings or floats hidden
    if 'bedrooms' in df.columns:
        numeric_beds = pd.to_numeric(df['bedrooms'], errors='coerce')
        nan_beds = numeric_beds.isna().sum()
        if nan_beds > 0:
            print(f"❌ BEDROOM ALERT: {nan_beds} listings have non-numeric or NaN bedroom values.")
            print("   Samples of bad values:", df[numeric_beds.isna()]['bedrooms'].unique()[:5])

    # Check if 'price' has 0 or negative values
    if 'price' in df.columns:
        zero_price = len(df[df['price'] <= 0])
        if zero_price > 0:
            print(f"⚠️ PRICE ALERT: {zero_price} listings have 0 or negative price.")

    # Check if 'property_type_id' matches 'property_type'
    if 'property_type' in df.columns and 'property_type_id' in df.columns:
        mismatches = df.groupby(['property_type', 'property_type_id']).size().reset_index(name='count')
        print("\nMapping Sample [Property Name -> ID]:")
        print(mismatches.sort_values('count', ascending=False).head(10).to_string(index=False))

    print("\n==================================================")
    print("🏁 QC COMPLETE")
    print("==================================================")

# Execute
perform_master_qc(df_listings)

📈 WORLD-CLASS DATA AUDIT REPORT

Total Listings: 44127
Total Columns: 45
--- 1. COLUMN SUMMARY ---
               Column          Dtype  Unique  Nulls Null %
  property_listing_id         object   44127      0   0.0%
          location_id         object    5316      0   0.0%
             latitude        float64    5224      0   0.0%
            longitude        float64    5211      0   0.0%
        location_name         object    5216      0   0.0%
   full_location_path         object    5316      0   0.0%
       property_title         object   41593      0   0.0%
     property_address        float64       0  44127 100.0%
        listing_level         object       3      0   0.0%
pending_verified_flag          int64       1      0   0.0%
         listing_date datetime64[ns]   21698      0   0.0%
        property_type         object      25      0   0.0%
             bedrooms        float64       8      0   0.0%
       bedrooms_label         object       9   2982   6.8%
            bath

In [28]:
# --- Forensics: Column Name Investigation ---
print("--- 🔍 COLUMN NAME FORENSICS ---")

# 1. Check for hidden spaces or special characters
raw_cols = df_listings.columns.tolist()
print(f"Raw Column List: {raw_cols}")

# 2. Check for spaces using repr() which shows ' ' clearly
print("\nRepresentation of names (looking for spaces):")
print([repr(col) for col in raw_cols if 'location' in col.lower()])

# 3. Check for duplicates (Pandas sometimes adds .1, .2)
if df_listings.columns.duplicated().any():
    print("\n⚠️ DUPLICATE COLUMNS FOUND:")
    print(df_listings.columns[df_listings.columns.duplicated()])
else:
    print("\n✅ No duplicate columns.")

--- 🔍 COLUMN NAME FORENSICS ---
Raw Column List: ['property_listing_id', 'location_id', 'latitude', 'longitude', 'location_name', 'full_location_path', 'property_title', 'property_address', 'listing_level', 'pending_verified_flag', 'listing_date', 'property_type', 'bedrooms', 'bedrooms_label', 'bathrooms', 'size_sqft', 'completion_status', 'furnished_flag', 'offering_type', 'category_id', 'property_type_id', 'price', 'price_period', 'quality_score', 'super_agent_score', 'amenities', 'view_count', 'save_count', 'lead_click_count', 'lead_submission_count', 'popularity_score', 'is_sale', 'price_period_clean', 'rent_multiplier', 'feature_annual_rent', 'feature_sale_price', 'log_price', 'listing_level_score', 'days_active', 'price_per_sqft', 'images', 'video_url', 'view_360', 'agent_name', 'broker_name']

Representation of names (looking for spaces):
["'location_id'", "'location_name'", "'full_location_path'"]

✅ No duplicate columns.


In [31]:
import pandas as pd

# Load the file
df = pd.read_parquet("artifacts/inventory.parquet")

# 1. Print all column names
print("--- ALL COLUMNS ---")
print(df.columns.tolist())

# 2. Print a sample row to see the data format
print("\n--- DATA SAMPLE (1st Row) ---")
print(df.iloc[0].to_dict())

# 3. Check for specific problematic columns
search_cols = ['location_name', 'location_name_en', 'property_title', 'title_en', 'id', 'property_id']
print("\n--- KEY COLUMN CHECK ---")
for col in search_cols:
    print(f"{col}: {'EXISTS' if col in df.columns else 'MISSING'}")

--- ALL COLUMNS ---
['property_listing_id', 'location_id', 'agent_id', 'broker_id', 'property_title', 'property_address', 'latitude', 'longitude', 'price', 'price_period', 'currency', 'images', 'video_url', 'view_360', 'agent_name', 'broker_name', 'is_super_agent', 'verified', 'listing_level', 'bedrooms', 'bathrooms', 'size_sqft', 'amenities', 'category_id', 'property_type_id', 'property_type', 'completion_status', 'furnished_flag', 'feature_annual_rent', 'feature_sale_price', 'is_sale', 'listing_level_score', 'super_agent_score', 'popularity_score', 'price_per_sqft', 'days_active']

--- DATA SAMPLE (1st Row) ---
{'property_listing_id': '14810182', 'location_id': '2689', 'agent_id': '0', 'broker_id': None, 'property_title': 'Spacious 1BDR | Close to Metro | with Balcony ', 'property_address': nan, 'latitude': 25.04451891, 'longitude': 55.13729112, 'price': 70000.0, 'price_period': 'yearly', 'currency': None, 'images': '', 'video_url': '', 'view_360': '', 'agent_name': '', 'broker_name'

In [35]:
import pandas as pd
df = pd.read_parquet("artifacts/inventory.parquet")

# Look at the amenities string for the first 10 properties that actually have them
sample_amenities = df[df['amenities'].str.len() > 0]['amenities'].head(10).tolist()

print("--- ACTUAL AMENITY STRINGS IN YOUR FILE ---")
for i, val in enumerate(sample_amenities):
    print(f"{i+1}: {val}")

--- ACTUAL AMENITY STRINGS IN YOUR FILE ---
1: BA,BK,BL,BW,EX,PG,PP,PY,SP,SY,VC,VW,WC
2: SY,VW,WC
3: BA,SP,SY
4: AC,BA,BK,BR,BW,CS,PR,SE,SP,SY
5: AC,BW,CP,PR
6: BL,BW,CO,CP
7: BK,BW,CO,CP,CS,LB,SE,SP,SY
8: BL,CP,MS,PY,SE
9: BA,BW
10: AC,BK,BW,CP,CS,LB,SE,SP,SY,WC


In [37]:
import pandas as pd

# Load your interaction data
if os.path.exists("artifacts/user_data.parquet"):
    df_user = pd.read_parquet("artifacts/user_data.parquet")
    # Get a user who has at least 3 interactions for better testing
    active_users = df_user['user_id'].value_counts()
    top_user = active_users.index[0]
    print(f"--- VALID TEST USER ---")
    print(f"User ID: {top_user}")
    print(f"Total Interactions: {active_users.iloc[0]}")
else:
    print("User interaction file not found. Try 'verify_user_01'.")

--- VALID TEST USER ---
User ID: a476a316-c3e6-496d-b84b-7c97d3b771b8
Total Interactions: 912


In [41]:
import pandas as pd

# 1. Load data
inventory = pd.read_parquet('artifacts/inventory.parquet')
user_data = pd.read_parquet('artifacts/user_data.parquet')

print("--- LISTINGS DATA (inventory.parquet) ---")
print(f"Total Rows: {len(inventory)}")
print(inventory.dtypes)
print("\nSample IDs:", inventory['property_listing_id'].head(3).tolist())

print("\n" + "="*40 + "\n")

print("--- USER DATA (user_data.parquet) ---")
print(f"Total Rows: {len(user_data)}")
print(user_data.dtypes)
print("\nSample IDs:", user_data['property_listing_id'].head(3).tolist())

# 2. Check for matching column names
listings_cols = set(inventory.columns)
user_cols = set(user_data.columns)
common_cols = listings_cols.intersection(user_cols)

print("\n" + "="*40 + "\n")
print(f"Common Columns found in both: {common_cols}")

# 3. Test the "Broken Bridge"
if 'property_listing_id' in common_cols:
    id_type_listing = inventory['property_listing_id'].dtype
    id_type_user = user_data['property_listing_id'].dtype
    
    print(f"\nID Type listing: {id_type_listing}")
    print(f"ID Type user:    {id_type_user}")
    
    if id_type_listing != id_type_user:
        print("\n❌ CRITICAL MISMATCH: One ID is a string and the other is a number.")
    else:
        print("\n✅ Types match, but are the values actually present in both?")
        
    # Check intersection count
    matches = set(inventory['property_listing_id'].astype(str)).intersection(set(user_data['property_listing_id'].astype(str)))
    print(f"Intersection Count (forced string match): {len(matches)}")

--- LISTINGS DATA (inventory.parquet) ---
Total Rows: 44127
property_listing_id     object
location_id             object
agent_id                object
broker_id               object
property_title          object
property_address       float64
latitude               float64
longitude              float64
price                  float64
price_period            object
currency                object
images                  object
video_url               object
view_360                object
agent_name              object
broker_name             object
is_super_agent          object
verified                object
listing_level           object
location_name           object
full_location_path      object
bedrooms               float64
bathrooms                int64
size_sqft                int64
amenities               object
category_id              int64
property_type_id         int64
property_type           object
completion_status        int64
furnished_flag           int64
feature_an